# Error bars for evals: hands-on (optional)
**LLMs, Animated · Lesson M4.6**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

You'll put error bars on an accuracy two ways (formula and bootstrap), compare two models the right way (**paired**), simulate how often a truly better model *loses* by chance, and build a sample-size calculator.

Stuck? Solutions are at the very bottom. Only `numpy` and `matplotlib` are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statistics import NormalDist   # standard library: normal quantiles
print("ready")

## 1 · Standard error and a 95% CI
An eval gives one 0/1 score per question. For accuracy $\hat p$ on $n$ questions:

$$\mathrm{SE} = \sqrt{\frac{\hat p(1-\hat p)}{n}}, \qquad \text{95\% CI} = \hat p \pm 1.96\,\mathrm{SE}$$

Return `(p_hat, se, lo, hi)`.

In [ ]:
def accuracy_ci(correct, z=1.96):
    # TODO: correct is an array of 0/1 scores
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's worked example: 400 right out of 500.

In [ ]:
scores = np.array([1] * 400 + [0] * 100)
p, se, lo, hi = accuracy_ci(scores)
assert abs(p - 0.8) < 1e-12
assert abs(se - 0.017889) < 1e-5
assert abs(lo - 0.76494) < 1e-4 and abs(hi - 0.83506) < 1e-4
print(f"accuracy {p:.1%}, SE {se:.2%}, 95% CI [{lo:.1%}, {hi:.1%}]")
print("✓ exercise 1")

## 2 · Bootstrap CI
Resample the $n$ questions **with replacement**, recompute the metric, repeat `n_boot` times, and return the 2.5th and 97.5th percentiles. Write it for any `metric` function, so it works beyond accuracy.

In [ ]:
def bootstrap_ci(correct, n_boot=2000, seed=0, metric=np.mean):
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct)
    n = len(correct)
    # TODO: draw n indices with rng.integers(0, n, n), compute metric on the resample, n_boot times
    raise NotImplementedError("Exercise 2")

**Check 2.** For a plain accuracy the bootstrap should agree with the formula.

In [ ]:
blo, bhi = bootstrap_ci(scores)
print(f"bootstrap 95% CI [{blo:.1%}, {bhi:.1%}]  vs formula [{lo:.1%}, {hi:.1%}]")
assert abs(blo - lo) < 0.01 and abs(bhi - hi) < 0.01
# 20 questions, 14 right: the lesson's animation (about 50% to 90%)
small = np.array([1] * 14 + [0] * 6)
slo, shi = bootstrap_ci(small, n_boot=5000)
assert 0.45 <= slo <= 0.55 and 0.85 <= shi <= 0.95
print("✓ exercise 2")

## 3 · Comparing two models: unpaired vs paired
The lesson's example: 1,000 questions. A is right on 820 and B on 800, and they disagree on 100 questions (A alone right on 60, B alone on 40).

Implement three functions, each returning a CI for the difference A − B:
- `unpaired_diff_ci`: treats the two scores as independent, $\mathrm{SE} = \sqrt{\mathrm{SE}_A^2 + \mathrm{SE}_B^2}$. Return `(d, se, lo, hi)`.
- `paired_diff_ci`: uses per-question differences $d_i = a_i - b_i$, $\mathrm{SE} = \mathrm{sd}(d)/\sqrt n$ (use `np.var` with its default `ddof=0`). Return `(d, se, lo, hi)`.
- `paired_bootstrap_ci`: resample question indices and score **both** models on the same resample. Return `(lo, hi)`.

In [ ]:
# Build the two score vectors: both right 760, A only 60, B only 40, both wrong 140
a = np.array([1] * 760 + [1] * 60 + [0] * 40 + [0] * 140)
b = np.array([1] * 760 + [0] * 60 + [1] * 40 + [0] * 140)
print("A:", a.mean(), " B:", b.mean())

In [ ]:
def unpaired_diff_ci(a, b, z=1.96):
    # TODO
    raise NotImplementedError("Exercise 3a")

def paired_diff_ci(a, b, z=1.96):
    # TODO
    raise NotImplementedError("Exercise 3b")

def paired_bootstrap_ci(a, b, n_boot=2000, seed=0):
    # TODO
    raise NotImplementedError("Exercise 3c")

**Check 3**

In [ ]:
d, se_u, ulo, uhi = unpaired_diff_ci(a, b)
_, se_p, plo, phi = paired_diff_ci(a, b)
bplo, bphi = paired_bootstrap_ci(a, b)
print(f"unpaired: {d:+.3f}  CI [{ulo:+.4f}, {uhi:+.4f}]")
print(f"paired  : {d:+.3f}  CI [{plo:+.4f}, {phi:+.4f}]")
print(f"paired bootstrap CI [{bplo:+.4f}, {bphi:+.4f}]")
assert abs(se_u - 0.017539) < 1e-5 and ulo < 0 < uhi, "separate CIs include 0"
assert abs(se_p - 0.009980) < 1e-5 and abs(plo - 0.00044) < 1e-4, "paired CI just excludes 0"
assert se_p < 0.6 * se_u
assert abs(bplo - plo) < 0.004 and abs(bphi - phi) < 0.004
mcnemar = (60 - 40) ** 2 / (60 + 40)
print("McNemar chi² =", mcnemar, " p ≈", round(2 * (1 - NormalDist().cdf(np.sqrt(mcnemar))), 3))
print("✓ exercise 3")

## 4 · How often does the better model lose?
Model A is truly better: 82% vs B's 80%. Both are evaluated on the **same-sized** benchmark of $n$ questions, but (to keep it simple) on independent draws of questions. In what fraction of evals does B score **at least as high** as A?

Use `rng.binomial(n, p, trials) / n` to simulate many evals at once.

In [ ]:
def flip_rate(p_a, p_b, n, trials=20000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO: simulate `trials` accuracies for each model and return the fraction where B >= A
    raise NotImplementedError("Exercise 4")

**Check 4.** The normal approximation predicts about 21% for $n = 500$.

In [ ]:
r500 = flip_rate(0.82, 0.80, 500)
print(f"n = 500: B ties or beats A in {r500:.1%} of evals")
assert 0.18 < r500 < 0.27
ns = [100, 300, 1000, 3000, 10000]
rates = [flip_rate(0.82, 0.80, n) for n in ns]
assert all(x > y for x, y in zip(rates, rates[1:])), "more questions, fewer flips"
plt.figure(figsize=(6, 3.2)); plt.semilogx(ns, rates, "o-")
plt.xlabel("questions per eval"); plt.ylabel("P(worse model ties or wins)"); plt.title("82% vs 80%: how often the ranking flips")
plt.grid(alpha=.3); plt.show()
print("✓ exercise 4")

## 5 · How many questions do you need?
For 80% power at the 5% level:

$$n \approx \frac{(z_{1-\alpha/2} + z_{\text{power}})^2 \cdot \mathrm{var}}{\delta^2}$$

with $\delta = p_1 - p_2$, and $\mathrm{var} = p_1(1-p_1) + p_2(1-p_2)$ for separate samples, or $\mathrm{var} = \text{discordance} - \delta^2$ for a paired design (discordance = fraction of questions where the models disagree). Use `NormalDist().inv_cdf` for the $z$ values and round **up**.

In [ ]:
def required_n(p1, p2, alpha=0.05, power=0.8, discordance=None):
    # TODO
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
n_unpaired = required_n(0.82, 0.80)
n_paired = required_n(0.82, 0.80, discordance=0.10)
print("unpaired:", n_unpaired, " paired:", n_paired, " 80% vs 85%:", required_n(0.85, 0.80))
assert 6000 <= n_unpaired <= 6100
assert 1940 <= n_paired <= 1970
assert 895 <= required_n(0.85, 0.80) <= 910
print("✓ exercise 5")

## Experiment: the multiple-comparisons trap
20 prompt variants, **all truly 80% accurate**, each scored on the same-sized set of 500 fresh questions. We pick the best one. How much does it seem to "improve" on the true 80%, and how often does it look significantly better than a baseline at 80%?

In [ ]:
rng = np.random.default_rng(42)
best_gain, false_wins = [], 0
for _ in range(2000):
    variants = rng.binomial(500, 0.80, 20) / 500
    base = rng.binomial(500, 0.80) / 500
    best = variants.max()
    best_gain.append(best - 0.80)
    se = np.sqrt(best * (1 - best) / 500 + base * (1 - base) / 500)
    false_wins += (best - base) / se > 1.96
print(f"average apparent gain of the best variant: {np.mean(best_gain) * 100:+.1f} points")
print(f"runs where the best variant 'significantly' beats the baseline: {false_wins / 2000:.0%}")
# Try: pick the best, then re-score it on 500 NEW questions. Where does it land?

## Recap
- Accuracy SE = $\sqrt{\hat p(1-\hat p)/n}$; the bootstrap gives CIs for any metric.
- Paired analysis on the same questions is far more sensitive than separate CIs.
- A 2-point gap near 80% needs thousands of questions to detect reliably.
- Picking the best of many noisy scores inflates it: re-test on fresh data.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def accuracy_ci(correct, z=1.96):
    correct = np.asarray(correct, dtype=float)
    n = len(correct)
    p = correct.mean()
    se = np.sqrt(p * (1 - p) / n)
    return p, se, p - z * se, p + z * se

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def bootstrap_ci(correct, n_boot=2000, seed=0, metric=np.mean):
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct)
    n = len(correct)
    stats = np.array([metric(correct[rng.integers(0, n, n)]) for _ in range(n_boot)])
    return np.percentile(stats, 2.5), np.percentile(stats, 97.5)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def unpaired_diff_ci(a, b, z=1.96):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a.mean() - b.mean()
    se = np.sqrt(a.mean() * (1 - a.mean()) / len(a) + b.mean() * (1 - b.mean()) / len(b))
    return d, se, d - z * se, d + z * se

def paired_diff_ci(a, b, z=1.96):
    d_i = np.asarray(a, float) - np.asarray(b, float)
    d = d_i.mean()
    se = np.sqrt(d_i.var() / len(d_i))     # var with ddof=0, as in the lesson
    return d, se, d - z * se, d + z * se

def paired_bootstrap_ci(a, b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    n = len(a)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)          # the SAME questions for both models
        diffs.append(a[idx].mean() - b[idx].mean())
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def flip_rate(p_a, p_b, n, trials=20000, seed=0):
    rng = np.random.default_rng(seed)
    acc_a = rng.binomial(n, p_a, trials) / n
    acc_b = rng.binomial(n, p_b, trials) / n
    return np.mean(acc_b >= acc_a)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def required_n(p1, p2, alpha=0.05, power=0.8, discordance=None):
    z = NormalDist().inv_cdf(1 - alpha / 2) + NormalDist().inv_cdf(power)
    delta = p1 - p2
    if discordance is None:
        var = p1 * (1 - p1) + p2 * (1 - p2)
    else:
        var = discordance - delta ** 2
    return int(np.ceil(z ** 2 * var / delta ** 2))